<a href="https://colab.research.google.com/github/ULTRONLORD/YorubaAIDetector/blob/main/Phase3_ModelTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 - Install all required libraries
!pip install transformers datasets scikit-learn pandas torch -q

print("✓ All libraries installed")

✓ All libraries installed


In [2]:
# CELL 2 - Import all required libraries

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)
import torch

# Confirm GPU is available
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

GPU available: True
Device: Tesla T4


In [3]:
# CELL 3 - Load dataset from Google Drive

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/YorubaAIDetector'

# Load your final dataset
df = pd.read_csv(f'{save_path}/yoruba_dataset_final.csv')

print("Dataset loaded successfully")
print("="*50)
print(f"Total samples:     {len(df)}")
print(f"Human-written (0): {len(df[df.label==0])}")
print(f"AI-generated  (1): {len(df[df.label==1])}")
print(f"\nSample text (first article):")
print(df.text.iloc[0][:200])

Mounted at /content/drive
Dataset loaded successfully
Total samples:     210
Human-written (0): 105
AI-generated  (1): 105

Sample text (first article):
"Ọ̀pọ̀ ìgbà ni àwọn ènìyàn ti gbà mi ni ìmọ̀ràn láti gbé òògùn fún ọmọ mi jẹ́ kó le bá lọ sinmi." Èyí ni ọ̀rọ̀ ìyá Faiza tó bi ọmọ tó ni ìpèníjà ọpọ̀lọ latinu ẹjẹ ta mọ si Autism. Nínú ìfọ̀rọ̀wérọ̀ rẹ


In [4]:
# CELL 4 - Split dataset into train, validation and test sets

# 70% training, 15% validation, 15% testing
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['label']  # ensures equal human/AI in each split
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label']
)

print("Dataset split complete")
print("="*50)
print(f"Training set:   {len(train_df)} articles")
print(f"Validation set: {len(val_df)} articles")
print(f"Test set:       {len(test_df)} articles")
print(f"\nTraining labels:")
print(train_df['label'].value_counts())S

Dataset split complete
Training set:   147 articles
Validation set: 31 articles
Test set:       32 articles

Training labels:
label
1    74
0    73
Name: count, dtype: int64


In [5]:
# CELL 5 - Load AfriBERTa tokenizer

MODEL_NAME = "castorini/afriberta_large"

print("Downloading AfriBERTa tokenizer...")
print("(This may take 2-3 minutes first time)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✓ Tokenizer loaded")

# Test it on one Yoruba sentence
test_sentence = "Ìjọba Nàìjíríà ti kéde eto tuntun"
tokens = tokenizer(test_sentence, return_tensors="pt")
print(f"\nTest tokenization:")
print(f"Input: {test_sentence}")
print(f"Token count: {tokens['input_ids'].shape[1]}")

(This may take 2-3 minutes first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✓ Tokenizer loaded

Test tokenization:
Input: Ìjọba Nàìjíríà ti kéde eto tuntun
Token count: 8


In [6]:
# CELL 6 - Tokenize all articles

def tokenize_data(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256  # 256 is enough for news articles
    )

# Convert pandas dataframes to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))

print("Tokenizing datasets...")

train_dataset = train_dataset.map(tokenize_data, batched=True)
val_dataset = val_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

print("✓ All datasets tokenized")
print(f"\nTraining dataset columns: {train_dataset.column_names}")

Tokenizing datasets...


Map:   0%|          | 0/147 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

✓ All datasets tokenized

Training dataset columns: ['text', 'label', 'input_ids', 'attention_mask']


In [7]:
# CELL 7 - Load AfriBERTa model for binary classification

print("Downloading AfriBERTa model...")
print("(This may take 3-5 minutes)")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # binary: human or AI
)

print("✓ Model loaded")
print(f"Model parameters: {model.num_parameters():,}")

(This may take 3-5 minutes)


pytorch_model.bin:   0%|          | 0.00/503M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/503M [00:00<?, ?B/s]

XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded
Model parameters: 125,632,514


In [8]:
# CELL 8 - Define how we measure performance

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    accuracy = accuracy_score(labels, predictions)

    return {
        'accuracy': round(accuracy, 4),
        'f1': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4)
    }

print("✓ Evaluation metrics defined")

✓ Evaluation metrics defined


In [10]:
# CELL 9 - Set up training configuration

training_args = TrainingArguments(
    output_dir='./yoruba_detector_model',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',        # renamed from evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_dir='./logs',
    logging_steps=10,
    warmup_steps=10,
    weight_decay=0.01,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("✓ Trainer configured")
print("Ready to train")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✓ Trainer configured
Ready to train


In [11]:
# CELL 10 - TRAIN THE MODEL
# This will take approximately 5-10 minutes on GPU
# You will see progress for each epoch - DO NOT interrupt it

print("Starting training...")
print("="*50)

trainer.train()

print("\n✓ Training complete!")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.661382,0.047628,1.000000,1.000000,1.000000,1.000000
2,0.149130,0.039421,0.967700,0.965500,1.000000,0.933300
3,0.001400,0.000434,1.000000,1.000000,1.000000,1.000000
4,0.003513,0.000351,1.000000,1.000000,1.000000,1.000000
5,0.000545,0.000328,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


✓ Training complete!
